In [1]:
import numpy
import pandas
import matplotlib
import seaborn
import sklearn
import xgboost

print("All packages imported successfully!")


All packages imported successfully!


In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from joblib import parallel_backend

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, roc_curve
)
from xgboost import XGBClassifier

# =============================================================
# WEEK 13 — CROSS-CITY EVALUATION
# WEEK 14 — URBAN FEATURE INFLUENCE + STATISTICAL TESTS
# =============================================================
#
# TARGET: extreme_rain column (0 = not extreme, 1 = extreme)
#         Binary classification. Extreme = top 5% daily rainfall per city.
#
# -----------------------------------------------------------
# MODELS AND THEIR EXACT FEATURE SETS:
# -----------------------------------------------------------
#
# MODEL 1: A_Full_RF — Random Forest, NO urban/LCZ features
#   Features:
#     Rainfall:  rainfall, rain_lag_1, rain_lag_2, rain_lag_3,
#                rain_roll_3, rain_roll_7
#     Weather:   tavg, tmin, tmax, pres, wspd
#     Time:      month, day_of_year, year
#     Geo:       x, y
#   NO building density. NO LCZ. NO avg_LCZ_city.
#   Why: Best within-city performer (AUC=1.0). Shows how a
#        "perfect" home model fails without urban context abroad.
#        Baseline for thesis argument.
#
# MODEL 2: B_Full_XGB — XGBoost, WITH urban + LCZ, WITH rainfall
#   Features: everything in A_Full PLUS:
#     Urban:     city_area_km2, building_count, buildings_per_km2,
#                total_building_area_km2, building_area_density
#     LCZ:       avg_LCZ_city, LCZ_0.0, LCZ_2.0 ... (all zones in train city)
#   Why: Tests if urban structure + LCZ features help XGB generalize.
#        Week 14 Q1 answer. Clean comparison vs A_Full_RF.
#
# MODEL 3: B_NoLags_XGB — XGBoost, WITH urban + LCZ, NO rainfall
#   Features: weather + time + geo PLUS all urban + LCZ
#   NO rainfall, NO lags, NO rolling averages.
#   Why: Realistic early-warning forecasting scenario.
#        Week 14 Q2: does rainfall memory transfer cross-city?
#        Hardest test — no rainfall signal at all.
#
# -----------------------------------------------------------
# LCZ ALIGNMENT STRATEGY (CORRECTED):
# -----------------------------------------------------------
#   Each city has different LCZ zone columns.
#   Singapore: LCZ_3.0 to LCZ_17.0  (11 zones)
#   Tokyo:     LCZ_0.0 to LCZ_17.0  (16 zones)
#   London:    LCZ_2.0 to LCZ_17.0  (8 zones)
#   New_York:  LCZ_0.0 to LCZ_17.0  (12 zones)
#
#   Strategy: Use ALL LCZ columns from the TRAINING city.
#   For the TEST city, any missing LCZ zones are filled with 0
#   (meaning that zone type doesn't exist in the test city).
#   This preserves the full LCZ information from training
#   and makes the cross-city comparison meaningful.
#
# -----------------------------------------------------------
# CITY PAIRS (6 total):
# -----------------------------------------------------------
#   Singapore -> Tokyo    tropical vs typhoon (biggest climate contrast)
#   Tokyo -> Singapore    reverse direction test
#   London -> New_York    temperate vs continental
#   New_York -> London    reverse direction test
#   Singapore -> London   tropical vs maritime (strong LCZ contrast)
#   Tokyo -> New_York     both large dense cities (best-case transfer)
#
# -----------------------------------------------------------
# TOKYO SPEED OPTIMIZATION:
# -----------------------------------------------------------
#   Any pair involving Tokyo uses n_jobs=-1 and parallel backend
#   (same approach as single-city Tokyo code) to avoid slow runtime.
#
# -----------------------------------------------------------
# STATISTICAL TESTS (WEEK 14):
# -----------------------------------------------------------
#   Paired t-test across all 6 pairs:
#   1. A_Full_RF vs B_Full_XGB AUC scores — does LCZ+urban help?
#   2. B_Full_XGB vs B_NoLags_XGB AUC scores — does rainfall transfer?
#   p < 0.05 = statistically significant difference
#
# -----------------------------------------------------------
# OUTPUT:
#   Results\tables.txt       — all metrics, tables, p-values
#   Results\figures\*.png    — all 7 visualizations
#   Results\cross_city_results.csv — raw results table
# =============================================================


# =========================
# 1. PATHS
# =========================

BASE        = r"C:\Users\janaa\UrbanRainfall-ML-Thesis"
RESULTS_DIR = rf"{BASE}\Results"
FIGURES_DIR = rf"{RESULTS_DIR}\figures"
TABLES_FILE = rf"{RESULTS_DIR}\tables.txt"

os.makedirs(FIGURES_DIR, exist_ok=True)

city_files = {
    "Singapore": rf"{BASE}\Data\Processed\Singapore_week7_ready.csv",
    "Tokyo":     rf"{BASE}\Data\Processed\Tokyo_week7_ready.csv",
    "London":    rf"{BASE}\Data\Processed\London_week7_ready.csv",
    "New_York":  rf"{BASE}\Data\Processed\New_York_week7_ready.csv",
}

tables_out = open(TABLES_FILE, "w", encoding="utf-8")

def log(text=""):
    print(text)
    tables_out.write(text + "\n")

def save_fig(name):
    path = rf"{FIGURES_DIR}\{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {name}.png")


# =========================
# 2. LOAD ALL CITIES
# =========================

city_data = {}
log("=== LOADING CITIES ===\n")
for city, path in city_files.items():
    df = pd.read_csv(path)
    df = df.sort_values(["year", "day_of_year"]).reset_index(drop=True)
    city_data[city] = df
    log(f"{city}: {len(df):,} rows | "
        f"extreme_rain rate: {df['extreme_rain'].mean():.3f} | "
        f"extreme days: {int(df['extreme_rain'].sum()):,}")


# =========================
# 3. FEATURE SETS
# =========================

def get_features(df):
    """
    Defines the three feature sets.

    A_Full  — rainfall + weather + time + geo
              NO urban building stats, NO LCZ
    B_Full  — A_Full + urban building stats + LCZ
    B_NoLags— weather + time + geo + urban + LCZ (no rainfall at all)

    Building density and LCZ are ONLY in B sets.
    avg_LCZ_city included in B sets (present in all cities).
    LCZ one-hot columns (LCZ_X.0) detected automatically.
    grid_id excluded.
    """
    rain_cols    = ["rainfall", "rain_lag_1", "rain_lag_2", "rain_lag_3",
                    "rain_roll_3", "rain_roll_7"]
    weather_cols = ["tavg", "tmin", "tmax", "pres", "wspd"]
    time_cols    = ["month", "day_of_year", "year"]
    geo_cols     = ["x", "y"]

    # Urban + LCZ — B sets only
    urban_cols   = ["city_area_km2", "building_count", "buildings_per_km2",
                    "total_building_area_km2", "building_area_density"]
    lcz_onehot   = [c for c in df.columns if c.startswith("LCZ_")]
    lcz_cols     = ["avg_LCZ_city"] + lcz_onehot

    base_no_urban = weather_cols + time_cols + geo_cols
    urban_lcz     = urban_cols + lcz_cols

    return {
        # Model 1: RF — no urban, no LCZ, full rainfall
        "A_Full":   rain_cols + base_no_urban,

        # Model 2: XGB — urban + LCZ + full rainfall
        "B_Full":   rain_cols + base_no_urban + urban_lcz,

        # Model 3: XGB — urban + LCZ, NO rainfall
        "B_NoLags": base_no_urban + urban_lcz,
    }


# =========================
# 4. LCZ ALIGNMENT — FILL MISSING WITH 0
# =========================

def align_features(X_train, X_test, train_city, test_city):
    """
    Uses ALL LCZ columns from the training city.
    If a zone exists in train but not in test -> fill test with 0.
    If a zone exists in test but not in train -> drop from test.
    This preserves the full LCZ feature set from training.
    """
    # Columns train has but test doesn't -> add to test as 0
    for col in X_train.columns:
        if col not in X_test.columns:
            X_test = X_test.copy()
            X_test[col] = 0
            print(f"    {col} missing in {test_city} -> filled with 0")

    # Columns test has but train hasn't seen -> drop from test
    extra_in_test = [c for c in X_test.columns if c not in X_train.columns]
    if extra_in_test:
        X_test = X_test.drop(columns=extra_in_test)
        print(f"    Dropped from test (unseen in train): {extra_in_test}")

    # Reorder test columns to match train exactly
    X_test = X_test[X_train.columns]

    print(f"    Features used: {len(X_train.columns)}")
    return X_train, X_test


# =========================
# 5. MODEL BUILDERS
# Tokyo pairs get n_jobs=-1 + parallel backend for speed
# =========================

def build_model(model_type, scale_pos_weight, use_tokyo_speed):
    if model_type == "RF":
        return RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1 if use_tokyo_speed else 1
        )
    elif model_type == "XGB":
        return XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            n_jobs=-1 if use_tokyo_speed else 1
        )


# =========================
# 6. CROSS-CITY RUN FUNCTION
# =========================

cross_results = []

def run_cross(train_city, test_city, feature_key, model_type):
    """
    Trains on first 80% of train_city.
    Tests on FULL test_city.
    Scaler fit ONLY on train_city data.
    LCZ: train city keeps all its zones, test city fills missing with 0.
    Tokyo pairs use parallel backend for speed.
    """
    use_tokyo_speed = ("Tokyo" in train_city or "Tokyo" in test_city)

    df_train = city_data[train_city]
    df_test  = city_data[test_city]

    feat_train = get_features(df_train)[feature_key]
    feat_test  = get_features(df_test)[feature_key]

    split_idx = int(len(df_train) * 0.8)
    X_train   = df_train[feat_train].iloc[:split_idx]
    y_train   = df_train["extreme_rain"].iloc[:split_idx]

    X_test = df_test[feat_test]
    y_test = df_test["extreme_rain"]

    # Align — fill missing LCZ zones with 0
    X_train, X_test = align_features(X_train, X_test, train_city, test_city)

    # Scale — fit ONLY on train city
    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Imbalance ratio from train city
    spw = (y_train == 0).sum() / (y_train == 1).sum()

    model = build_model(model_type, spw, use_tokyo_speed)

    if use_tokyo_speed:
        with parallel_backend("loky"):
            model.fit(X_train_s, y_train)
    else:
        model.fit(X_train_s, y_train)

    preds = model.predict(X_test_s)
    probs = model.predict_proba(X_test_s)[:, 1]

    auc  = roc_auc_score(y_test, probs)
    f1   = f1_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    prec = precision_score(y_test, preds, zero_division=0)
    acc  = accuracy_score(y_test, preds)

    cross_results.append({
        "Train":     train_city,
        "Test":      test_city,
        "Pair":      f"{train_city}->{test_city}",
        "Model":     f"{feature_key}_{model_type}",
        "Feature":   feature_key,
        "ModelType": model_type,
        "Accuracy":  round(acc,  6),
        "Precision": round(prec, 6),
        "Recall":    round(rec,  6),
        "F1":        round(f1,   6),
        "ROC_AUC":   round(auc,  6),
    })

    print(f"  Done {train_city}->{test_city} | {feature_key}_{model_type} | "
          f"AUC={auc:.4f} | F1={f1:.4f} | Recall={rec:.4f}")

    return model, X_test_s, y_test, preds, probs


# =========================
# 7. RUN ALL 6 PAIRS x 3 MODELS = 18 RUNS
# =========================

pairs = [
    ("Singapore", "Tokyo"),    # tropical -> typhoon
    ("Tokyo",     "Singapore"),# reverse
    ("London",    "New_York"), # temperate -> continental
    ("New_York",  "London"),   # reverse
    ("Singapore", "London"),   # tropical -> maritime
    ("Tokyo",     "New_York"), # large dense cities
]

models_to_run = [
    ("A_Full",   "RF"),   # Model 1
    ("B_Full",   "XGB"),  # Model 2
    ("B_NoLags", "XGB"),  # Model 3
]

log("\n=== RUNNING ALL PAIRS (WEEK 13) ===\n")
log("6 city pairs x 3 models = 18 experiments\n")

for train_city, test_city in pairs:
    print(f"\n--- {train_city} -> {test_city} ---")
    for feature_key, model_type in models_to_run:
        run_cross(train_city, test_city, feature_key, model_type)


# =========================
# 8. RESULTS TABLE — WEEK 13
# =========================

cross_df = pd.DataFrame(cross_results)

log("\n\n" + "="*70)
log("WEEK 13 — CROSS-CITY RESULTS (sorted by ROC_AUC descending)")
log("="*70)
log(cross_df.sort_values("ROC_AUC", ascending=False).to_string(index=False))

cross_df.to_csv(rf"{RESULTS_DIR}\cross_city_results.csv", index=False)
log("\nSaved: cross_city_results.csv")

log("\n--- Results by Model (average across all pairs) ---")
log(cross_df.groupby("Model")[["Accuracy","Precision","Recall","F1","ROC_AUC"]]
    .mean().round(4).to_string())


# =========================
# 9. PERFORMANCE DROP vs WITHIN-CITY
# =========================

# Your exact within-city AUC scores from Month 3
within_city_auc = {
    ("Singapore", "A_Full_RF"):    1.000000,
    ("Singapore", "B_Full_XGB"):   0.999962,
    ("Singapore", "B_NoLags_XGB"): 0.782884,
    ("Tokyo",     "A_Full_RF"):    1.000000,
    ("Tokyo",     "B_Full_XGB"):   0.999971,
    ("Tokyo",     "B_NoLags_XGB"): 0.808006,
    ("London",    "A_Full_RF"):    1.000000,
    ("London",    "B_Full_XGB"):   0.999990,
    ("London",    "B_NoLags_XGB"): 0.818750,
    ("New_York",  "A_Full_RF"):    1.000000,
    ("New_York",  "B_Full_XGB"):   0.999996,
    ("New_York",  "B_NoLags_XGB"): 0.835906,
}

drop_rows = []
for _, row in cross_df.iterrows():
    key    = (row["Train"], f"{row['Feature']}_{row['ModelType']}")
    within = within_city_auc.get(key)
    if within is not None:
        drop_rows.append({
            "Pair":       row["Pair"],
            "Model":      row["Model"],
            "Within_AUC": within,
            "Cross_AUC":  row["ROC_AUC"],
            "AUC_Drop":   round(within - row["ROC_AUC"], 4),
        })

drop_df = pd.DataFrame(drop_rows).sort_values("AUC_Drop", ascending=False)

log("\n\n" + "="*70)
log("WEEK 13 — PERFORMANCE DROP: WITHIN-CITY vs CROSS-CITY")
log("="*70)
log(drop_df.to_string(index=False))

log("\nAverage AUC Drop by Model:")
log(drop_df.groupby("Model")["AUC_Drop"].mean().round(4).to_string())
log("Higher = model generalizes worse across cities")


# =============================================================
# WEEK 14 — URBAN FEATURE INFLUENCE ANALYSIS
# =============================================================

log("\n\n" + "="*70)
log("WEEK 14 — URBAN FEATURE INFLUENCE ANALYSIS")
log("="*70)

# -------------------------
# Q1: Does LCZ + building density help cross-city generalization?
# A_Full_RF (no urban/LCZ) vs B_Full_XGB (with urban + LCZ)
# Note: model type also differs — attribute findings carefully
# -------------------------

log("\n--- Q1: Does LCZ + building density help cross-city? ---")
log("A_Full_RF = no urban features, no LCZ")
log("B_Full_XGB = with building density + avg_LCZ_city + LCZ zone columns")
log("Note: model type also differs (RF vs XGB) — interpret carefully\n")

lcz_q1 = cross_df[cross_df["Feature"].isin(["A_Full","B_Full"])].copy()
lcz_pivot = lcz_q1.pivot_table(
    index="Pair", columns="Feature", values="ROC_AUC"
).reset_index()
lcz_pivot.columns.name = None
lcz_pivot = lcz_pivot.rename(columns={
    "A_Full": "AUC_NoUrban_RF",
    "B_Full": "AUC_Urban_LCZ_XGB"
})
lcz_pivot["LCZ_Urban_Benefit"] = (
    lcz_pivot["AUC_Urban_LCZ_XGB"] - lcz_pivot["AUC_NoUrban_RF"]
).round(4)

log(lcz_pivot.to_string(index=False))
log(f"\nMean benefit across all pairs: {lcz_pivot['LCZ_Urban_Benefit'].mean():.4f}")
log("Positive = LCZ + urban features + XGB better cross-city")
log("Negative = plain RF without urban features generalizes better")


# -------------------------
# Q2: Does rainfall memory help cross-city?
# B_Full_XGB vs B_NoLags_XGB — CLEAN comparison
# EXACTLY same model, same LCZ, same urban — only rainfall differs
# -------------------------

log("\n\n--- Q2: Does rainfall memory transfer cross-city? ---")
log("B_Full_XGB   = XGB + urban + LCZ + full rainfall features")
log("B_NoLags_XGB = XGB + urban + LCZ + NO rainfall at all")
log("Same model, same urban structure — only rainfall features differ\n")

rain_q2 = cross_df[
    (cross_df["ModelType"] == "XGB") &
    (cross_df["Feature"].isin(["B_Full","B_NoLags"]))
].copy()
rain_pivot = rain_q2.pivot_table(
    index="Pair", columns="Feature", values="ROC_AUC"
).reset_index()
rain_pivot.columns.name = None
rain_pivot = rain_pivot.rename(columns={
    "B_Full":   "AUC_WithRainfall",
    "B_NoLags": "AUC_NoRainfall"
})
rain_pivot["Rainfall_Benefit"] = (
    rain_pivot["AUC_WithRainfall"] - rain_pivot["AUC_NoRainfall"]
).round(4)

log(rain_pivot.to_string(index=False))
log(f"\nMean benefit across all pairs: {rain_pivot['Rainfall_Benefit'].mean():.4f}")
log("Positive = rainfall memory helps even cross-city")
log("Negative = rainfall patterns are city-specific, don't transfer")


# -------------------------
# Q3: Does building density similarity predict generalization?
# -------------------------

log("\n\n--- Q3: Does urban density similarity predict AUC drop? ---")

density_stats = {}
for city, df in city_data.items():
    density_stats[city] = df["buildings_per_km2"].mean()
    log(f"  {city}: mean buildings_per_km2 = {density_stats[city]:.2f}")

drop_df["Train_Density"] = drop_df["Pair"].apply(
    lambda p: density_stats.get(p.split("->")[0], np.nan))
drop_df["Test_Density"] = drop_df["Pair"].apply(
    lambda p: density_stats.get(p.split("->")[1], np.nan))
drop_df["Density_Diff"] = abs(
    drop_df["Train_Density"] - drop_df["Test_Density"]).round(2)

log("\nAUC Drop vs Building Density Difference:")
log(drop_df[["Pair","Model","AUC_Drop","Density_Diff"]].sort_values(
    "Density_Diff", ascending=False).to_string(index=False))

corr = drop_df[["AUC_Drop","Density_Diff"]].corr().iloc[0,1]
log(f"\nCorrelation (AUC Drop vs Density Difference): {corr:.4f}")
log("Positive = more different cities -> bigger AUC drop")
log("Near zero = density difference alone doesn't predict failure")


# =========================
# STATISTICAL TESTS (WEEK 14)
# Paired t-test across the 6 pairs
# =========================

log("\n\n" + "="*70)
log("WEEK 14 — STATISTICAL SIGNIFICANCE TESTS (PAIRED T-TEST)")
log("="*70)
log("Hypothesis: cross-city AUC scores differ significantly between models")
log("Test: paired t-test across 6 city pairs")
log("Threshold: p < 0.05 = statistically significant\n")

# Get AUC per pair for each model (aligned by pair order)
pair_order = [f"{a}->{b}" for a, b in pairs]

def get_aucs_by_pair(model_name):
    sub = cross_df[cross_df["Model"] == model_name].set_index("Pair")
    return [sub.loc[p, "ROC_AUC"] for p in pair_order if p in sub.index]

aucs_rf     = get_aucs_by_pair("A_Full_RF")
aucs_b_full = get_aucs_by_pair("B_Full_XGB")
aucs_nolags = get_aucs_by_pair("B_NoLags_XGB")

# Test 1: A_Full_RF vs B_Full_XGB
# Does adding LCZ + urban features + using XGB significantly change AUC?
t1, p1 = stats.ttest_rel(aucs_rf, aucs_b_full)
log("--- Test 1: A_Full_RF vs B_Full_XGB ---")
log(f"  A_Full_RF   AUC scores: {[round(a,4) for a in aucs_rf]}")
log(f"  B_Full_XGB  AUC scores: {[round(a,4) for a in aucs_b_full]}")
log(f"  t-statistic: {t1:.4f}")
log(f"  p-value:     {p1:.4f}")
if p1 < 0.05:
    log("  Result: SIGNIFICANT (p < 0.05)")
    log("  Interpretation: LCZ + urban features + XGB produces")
    log("  statistically different cross-city AUC scores than RF alone.")
else:
    log("  Result: NOT SIGNIFICANT (p >= 0.05)")
    log("  Interpretation: No statistically significant difference between")
    log("  using LCZ + urban features vs not, across these city pairs.")

# Test 2: B_Full_XGB vs B_NoLags_XGB
# Does rainfall memory significantly improve cross-city AUC?
t2, p2 = stats.ttest_rel(aucs_b_full, aucs_nolags)
log("\n--- Test 2: B_Full_XGB vs B_NoLags_XGB ---")
log(f"  B_Full_XGB   AUC scores: {[round(a,4) for a in aucs_b_full]}")
log(f"  B_NoLags_XGB AUC scores: {[round(a,4) for a in aucs_nolags]}")
log(f"  t-statistic: {t2:.4f}")
log(f"  p-value:     {p2:.4f}")
if p2 < 0.05:
    log("  Result: SIGNIFICANT (p < 0.05)")
    log("  Interpretation: Rainfall memory significantly improves")
    log("  cross-city AUC. Rainfall features transfer meaningfully.")
else:
    log("  Result: NOT SIGNIFICANT (p >= 0.05)")
    log("  Interpretation: No significant difference from including")
    log("  rainfall features cross-city. Rainfall patterns may be city-specific.")

# Test 3: Within-city vs Cross-city AUC (all models combined)
# Does generalization significantly drop overall?
within_aucs = [within_city_auc.get((row["Train"], f"{row['Feature']}_{row['ModelType']}"))
               for _, row in cross_df.iterrows()]
cross_aucs  = list(cross_df["ROC_AUC"])
within_aucs = [w for w in within_aucs if w is not None]

t3, p3 = stats.ttest_rel(within_aucs, cross_aucs)
log("\n--- Test 3: Within-city AUC vs Cross-city AUC (all models) ---")
log(f"  Mean within-city AUC: {np.mean(within_aucs):.4f}")
log(f"  Mean cross-city AUC:  {np.mean(cross_aucs):.4f}")
log(f"  t-statistic: {t3:.4f}")
log(f"  p-value:     {p3:.4f}")
if p3 < 0.05:
    log("  Result: SIGNIFICANT (p < 0.05)")
    log("  Interpretation: Cross-city AUC is significantly lower than")
    log("  within-city AUC. Standard within-city evaluation significantly")
    log("  overestimates real-world generalization performance.")
    log("  This directly supports the thesis central argument.")
else:
    log("  Result: NOT SIGNIFICANT (p >= 0.05)")


# =============================================================
# VISUALIZATIONS
# =============================================================

print("\n=== GENERATING FIGURES ===\n")

# FIG 1: Heatmap per model
for model_name in ["A_Full_RF", "B_Full_XGB", "B_NoLags_XGB"]:
    subset = cross_df[cross_df["Model"] == model_name]
    pivot  = subset.pivot_table(index="Train", columns="Test", values="ROC_AUC")
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0.5, vmax=1.0, linewidths=0.5,
                annot_kws={"size": 12}, ax=ax)
    ax.set_title(f"Cross-City ROC-AUC Heatmap\n{model_name}", fontsize=13)
    ax.set_xlabel("Test City")
    ax.set_ylabel("Train City")
    plt.tight_layout()
    save_fig(f"heatmap_{model_name}")

# FIG 2: AUC drop bar chart
fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#e74c3c" if v > 0.15 else "#f39c12" if v > 0.05 else "#2ecc71"
          for v in drop_df["AUC_Drop"]]
x = range(len(drop_df))
ax.bar(x, drop_df["AUC_Drop"], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['Pair']}\n{r['Model']}" for _, r in drop_df.iterrows()],
    rotation=45, ha="right", fontsize=7)
ax.set_ylabel("AUC Drop (Within -> Cross-city)")
ax.set_title("Performance Drop: Within-City vs Cross-City\n"
             "Red = large (>0.15) | Amber = moderate (>0.05) | Green = small")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
save_fig("auc_drop_barchart")

# FIG 3: LCZ + urban benefit per pair (Q1)
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in lcz_pivot["LCZ_Urban_Benefit"]]
ax.bar(lcz_pivot["Pair"], lcz_pivot["LCZ_Urban_Benefit"], color=colors)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("AUC Difference\n(B_Full_XGB − A_Full_RF)")
ax.set_title("Week 14 Q1 — Does LCZ + Building Density Help Cross-City?\n"
             "Green = yes | Red = no")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
save_fig("lcz_urban_benefit_Q1")

# FIG 4: Rainfall memory benefit per pair (Q2)
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in rain_pivot["Rainfall_Benefit"]]
ax.bar(rain_pivot["Pair"], rain_pivot["Rainfall_Benefit"], color=colors)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("AUC Difference\n(B_Full_XGB − B_NoLags_XGB)")
ax.set_title("Week 14 Q2 — Does Rainfall Memory Transfer Cross-City?\n"
             "Same model + same LCZ — only rainfall differs\n"
             "Green = yes | Red = no")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
save_fig("rainfall_memory_benefit_Q2")

# FIG 5: Grouped bar — all 3 models per pair
pairs_list  = [f"{a}->{b}" for a, b in pairs]
models_list = ["A_Full_RF", "B_Full_XGB", "B_NoLags_XGB"]
x           = np.arange(len(pairs_list))
width       = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
for i, model_name in enumerate(models_list):
    subset = cross_df[cross_df["Model"] == model_name]
    aucs   = []
    for pair in pairs_list:
        row = subset[subset["Pair"] == pair]
        aucs.append(row["ROC_AUC"].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, aucs, width, label=model_name)

ax.set_xticks(x + width)
ax.set_xticklabels(pairs_list, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.4, 1.05)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Random baseline")
ax.set_title("Cross-City ROC-AUC — All 3 Models × All 6 Pairs")
ax.legend(fontsize=9)
plt.tight_layout()
save_fig("grouped_bar_all_models_pairs")

# FIG 6: ROC curves — best pair per model
best_per_model = cross_df.loc[cross_df.groupby("Model")["ROC_AUC"].idxmax()]
fig, ax = plt.subplots(figsize=(7, 6))

for _, row in best_per_model.iterrows():
    train_city  = row["Train"]
    test_city   = row["Test"]
    feature_key = row["Feature"]
    model_type  = row["ModelType"]
    use_tokyo   = ("Tokyo" in train_city or "Tokyo" in test_city)

    df_train = city_data[train_city]
    df_test  = city_data[test_city]

    feat_train = get_features(df_train)[feature_key]
    feat_test  = get_features(df_test)[feature_key]

    split_idx = int(len(df_train) * 0.8)
    X_train   = df_train[feat_train].iloc[:split_idx]
    y_train   = df_train["extreme_rain"].iloc[:split_idx]
    X_test    = df_test[feat_test]
    y_test    = df_test["extreme_rain"]

    X_train, X_test = align_features(X_train, X_test, train_city, test_city)

    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    spw   = (y_train == 0).sum() / (y_train == 1).sum()
    model = build_model(model_type, spw, use_tokyo)

    if use_tokyo:
        with parallel_backend("loky"):
            model.fit(X_train_s, y_train)
    else:
        model.fit(X_train_s, y_train)

    probs = model.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr,
            label=f"{row['Model']} | {row['Pair']} (AUC={row['ROC_AUC']:.3f})")

ax.plot([0,1],[0,1], "--", color="gray", label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Best Cross-City Pair Per Model")
ax.legend(fontsize=8)
plt.tight_layout()
save_fig("roc_curves_best_pairs")

# FIG 7: Scatter — density difference vs AUC drop (Q3)
fig, ax = plt.subplots(figsize=(7, 5))
for model_name in drop_df["Model"].unique():
    sub = drop_df[drop_df["Model"] == model_name]
    ax.scatter(sub["Density_Diff"], sub["AUC_Drop"], label=model_name, s=70)

ax.set_xlabel("Building Density Difference |train − test| (buildings/km²)")
ax.set_ylabel("AUC Drop (Within -> Cross-city)")
ax.set_title(f"Week 14 Q3 — Urban Similarity vs Generalization Drop\n"
             f"Pearson Correlation: {corr:.3f}")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.legend(fontsize=8)
plt.tight_layout()
save_fig("density_vs_auc_drop_Q3")


# =============================================================
# FINAL SUMMARY
# =============================================================

log("\n\n" + "="*70)
log("WEEK 13-14 FINAL SUMMARY")
log("="*70)

log(f"\nTotal experiments run: {len(cross_df)} (6 pairs x 3 models)")

log("\n--- Best cross-city result per model ---")
log(cross_df.loc[cross_df.groupby("Model")["ROC_AUC"].idxmax()][
    ["Model","Pair","ROC_AUC","F1","Recall"]].to_string(index=False))

log("\n--- Worst cross-city result per model ---")
log(cross_df.loc[cross_df.groupby("Model")["ROC_AUC"].idxmin()][
    ["Model","Pair","ROC_AUC","F1","Recall"]].to_string(index=False))

log("\n--- Average AUC drop (within -> cross-city) ---")
log(drop_df.groupby("Model")["AUC_Drop"].mean().round(4).to_string())

log("\n--- Statistical test summary ---")
log(f"Test 1 (RF vs XGB+LCZ+Urban): p = {p1:.4f} "
    f"{'SIGNIFICANT' if p1 < 0.05 else 'NOT SIGNIFICANT'}")
log(f"Test 2 (XGB Full vs No-lag):  p = {p2:.4f} "
    f"{'SIGNIFICANT' if p2 < 0.05 else 'NOT SIGNIFICANT'}")
log(f"Test 3 (Within vs Cross AUC): p = {p3:.4f} "
    f"{'SIGNIFICANT' if p3 < 0.05 else 'NOT SIGNIFICANT'}")

log(f"\n--- Q3 Urban similarity ---")
log(f"Pearson correlation (AUC drop vs density diff): {corr:.4f}")

tables_out.close()
print(f"\nAll tables saved to: {TABLES_FILE}")
print(f"All figures saved to: {FIGURES_DIR}")
print("\n=== DONE ===")

=== LOADING CITIES ===

Singapore: 657,120 rows | extreme_rain rate: 0.050 | extreme days: 32,873
Tokyo: 4,408,180 rows | extreme_rain rate: 0.050 | extreme days: 220,530
London: 651,644 rows | extreme_rain rate: 0.050 | extreme days: 32,600
New_York: 525,696 rows | extreme_rain rate: 0.050 | extreme days: 26,300

=== RUNNING ALL PAIRS (WEEK 13) ===

6 city pairs x 3 models = 18 experiments


--- Singapore -> Tokyo ---
    Features used: 16
  Done Singapore->Tokyo | A_Full_RF | AUC=0.9987 | F1=0.9742 | Recall=1.0000
    LCZ_13.0 missing in Tokyo -> filled with 0
    Dropped from test (unseen in train): ['LCZ_0.0', 'LCZ_1.0', 'LCZ_2.0', 'LCZ_5.0', 'LCZ_16.0']
    Features used: 33
  Done Singapore->Tokyo | B_Full_XGB | AUC=0.9990 | F1=0.9793 | Recall=1.0000
    LCZ_13.0 missing in Tokyo -> filled with 0
    Dropped from test (unseen in train): ['LCZ_0.0', 'LCZ_1.0', 'LCZ_2.0', 'LCZ_5.0', 'LCZ_16.0']
    Features used: 27
  Done Singapore->Tokyo | B_NoLags_XGB | AUC=0.5595 | F1=0.0851 | 

# Week 16 — Cross-City Evaluation Analysis
## Month 4 Results: Weeks 13 & 14

**Thesis:** Evaluating the Generalization of Machine Learning Models for Urban Extreme Rainfall Prediction
**Cities:** Singapore · London · New York · Tokyo
**Models:** A_Full_RF · B_Full_XGB · B_NoLags_XGB
**Experiments:** 6 city pairs × 3 models = 18 cross-city experiments

---

## Overview

Weeks 13 and 14 form the core of the thesis experimental contribution. Having established near-perfect within-city performance in Month 3, this phase tests how well those models actually generalize when applied to cities they have never seen. Six directional city pairs were selected to capture a range of climatic and urban contrasts, and three statistical tests were conducted to evaluate whether observed differences were statistically meaningful.

---

## Week 13 — Cross-City Evaluation

### Experimental Setup

Each model was trained on the first 80% of one city's data and tested on the entire dataset of a different city. The scaler was fit exclusively on training city data to prevent any information leakage from the test city. For LCZ features, a fill-with-zero strategy was used: any LCZ zone present in the training city but absent from the test city was filled with 0, preserving the full LCZ feature set from training while handling structural differences between cities.

The six city pairs tested were:

| Pair | Climate Contrast |
|------|----------------|
| Singapore → Tokyo | Tropical → typhoon-influenced |
| Tokyo → Singapore | Reverse direction |
| London → New York | Temperate maritime → continental |
| New York → London | Reverse direction |
| Singapore → London | Tropical → temperate maritime (extreme contrast) |
| Tokyo → New York | Both large dense cities (best-case transfer) |

### Results Table

| Pair | Model | AUC | F1 | Recall |
|------|-------|-----|-----|--------|
| Singapore→Tokyo | A_Full_RF | 0.999 | 0.974 | 1.000 |
| Singapore→Tokyo | B_Full_XGB | 0.999 | 0.979 | 1.000 |
| Singapore→Tokyo | B_NoLags_XGB | 0.560 | 0.085 | 0.068 |
| Tokyo→Singapore | A_Full_RF | 0.984 | 0.963 | 0.928 |
| Tokyo→Singapore | B_Full_XGB | 0.989 | 0.982 | 0.964 |
| Tokyo→Singapore | B_NoLags_XGB | 0.612 | 0.058 | 0.054 |
| London→New York | A_Full_RF | 0.975 | 0.624 | 1.000 |
| London→New York | B_Full_XGB | 0.974 | 0.625 | 1.000 |
| London→New York | B_NoLags_XGB | 0.651 | 0.007 | 0.004 |
| New York→London | A_Full_RF | 0.674 | 0.356 | 0.217 |
| New York→London | B_Full_XGB | 0.837 | 0.355 | 0.216 |
| New York→London | B_NoLags_XGB | 0.747 | 0.181 | 0.461 |
| Singapore→London | A_Full_RF | 0.554 | 0.183 | 0.101 |
| Singapore→London | B_Full_XGB | 0.433 | 0.178 | 0.097 |
| Singapore→London | B_NoLags_XGB | 0.473 | 0.000 | 0.000 |
| Tokyo→New York | A_Full_RF | 0.845 | 0.787 | 0.649 |
| Tokyo→New York | B_Full_XGB | 0.901 | 0.801 | 0.668 |
| Tokyo→New York | B_NoLags_XGB | 0.597 | 0.143 | 0.184 |

### Performance Drop: Within-City vs Cross-City

| Pair | Model | Within AUC | Cross AUC | Drop |
|------|-------|-----------|-----------|------|
| Singapore→London | B_Full_XGB | 1.000 | 0.433 | **0.567** |
| Singapore→London | A_Full_RF | 1.000 | 0.554 | **0.446** |
| New York→London | A_Full_RF | 1.000 | 0.674 | **0.326** |
| Singapore→London | B_NoLags_XGB | 0.783 | 0.473 | 0.310 |
| Singapore→Tokyo | B_NoLags_XGB | 0.783 | 0.560 | 0.223 |
| Tokyo→New York | B_NoLags_XGB | 0.808 | 0.597 | 0.211 |
| Singapore→Tokyo | A_Full_RF | 1.000 | 0.999 | 0.001 |
| Singapore→Tokyo | B_Full_XGB | 1.000 | 0.999 | 0.001 |

Average AUC drop by model:
- A_Full_RF: 0.162
- B_Full_XGB: 0.145
- B_NoLags_XGB: 0.200

### Visualization Interpretation

**Heatmaps (A_Full_RF, B_Full_XGB, B_NoLags_XGB)**

The three heatmaps reveal a striking asymmetry in cross-city performance. London as a test city consistently produces the worst results — Singapore→London is deep red (AUC 0.554 for RF, 0.433 for XGB) across both full-feature models, while Singapore→Tokyo and Tokyo→Singapore remain dark green (AUC 0.983–0.999). This visual contrast is the clearest evidence that generalization failure is not random — it is systematic and tied to specific city pairs. The B_NoLags_XGB heatmap is almost entirely orange and red, confirming that removing rainfall features universally degrades cross-city performance regardless of which cities are involved.

**AUC Drop Bar Chart**

The bar chart ordered by drop size shows Singapore→London dominating the left side in deep red — the single largest generalization failures in the study. The right side transitions through amber to green, with Singapore→Tokyo and Tokyo→Singapore barely visible. The visual reinforces that some city pairs transfer almost without loss while others collapse entirely. Notably, all Singapore→London bars are red regardless of model type, confirming this is a city-pair problem rather than a model problem.

**Grouped Bar Chart (All Models × All Pairs)**

This chart most clearly shows the consistent structure across models. For Singapore→Tokyo and Tokyo→Singapore, A_Full_RF and B_Full_XGB are both near 1.0 while B_NoLags_XGB drops to ~0.56–0.61. For Singapore→London, all three models are low — A_Full_RF at 0.554, B_Full_XGB at 0.433, B_NoLags_XGB at 0.473 — showing that no model could rescue this pair. For New York→London, B_Full_XGB (0.837) visibly outperforms A_Full_RF (0.674), the clearest single-pair evidence of LCZ and urban features adding value.

**ROC Curves (Best Pair Per Model)**

The ROC curves show A_Full_RF and B_Full_XGB both achieving near-perfect curves for Singapore→Tokyo — their lines overlap and hug the top-left corner indistinguishably. B_NoLags_XGB for its best pair (New York→London, AUC 0.747) shows a clearly inferior curve — rising more gradually, with meaningful area below the perfect boundary. The gap between the two curve shapes visually communicates the cost of removing rainfall memory and the ceiling imposed by using only weather and urban features cross-city.

### Key Patterns — Week 13

**1. Generalization is highly pair-dependent**
Singapore→Tokyo dropped only 0.001 AUC. Singapore→London dropped 0.446–0.567. The same model behaves completely differently depending on which cities are involved. This means "cross-city performance" is not a single number — it varies enormously by climate pair.

**2. Direction matters**
London→New York achieved AUC 0.975 but New York→London achieved only 0.674 with the same model. Training on a city with more variable or intense rainfall helps the model handle diverse test conditions. Training on London's moderate consistent rainfall leaves the model unprepared for the greater variability of other cities.

**3. Singapore→London is the thesis's key failure case**
This pair represents the most extreme climatic contrast — tropical monsoon versus temperate maritime. Every model fails here. B_Full_XGB with all urban and LCZ features drops to AUC 0.433 — worse than random guessing. This is the clearest single demonstration that neither model sophistication nor urban features can compensate for fundamental climate regime mismatch.

**4. B_NoLags_XGB consistently worst — but informative**
Without rainfall memory, every pair degrades substantially. Mean AUC 0.606 vs 0.839 for A_Full_RF. However B_NoLags maintains above-random performance in most pairs, confirming that weather and urban features retain some cross-city signal even without rainfall history.

---

## Week 14 — Urban Feature Influence Analysis

### Q1: Does LCZ + Building Density Help Cross-City Generalization?

Comparing A_Full_RF (no urban features, no LCZ) against B_Full_XGB (with building density + LCZ) across all six pairs:

| Pair | A_Full_RF AUC | B_Full_XGB AUC | Benefit |
|------|-------------|--------------|---------|
| London→New York | 0.975 | 0.974 | -0.001 |
| New York→London | 0.674 | 0.837 | **+0.164** |
| Singapore→London | 0.554 | 0.433 | **-0.121** |
| Singapore→Tokyo | 0.999 | 0.999 | +0.000 |
| Tokyo→New York | 0.845 | 0.901 | +0.055 |
| Tokyo→Singapore | 0.984 | 0.989 | +0.005 |

Mean benefit: +0.017 (p = 0.670 — NOT SIGNIFICANT)

**Visualization Interpretation (Q1 Bar Chart)**

The Q1 bar chart shows a mixed picture with no consistent direction. New York→London shows the largest green bar (+0.164) — the clearest case where LCZ and urban features helped. Singapore→London shows the largest red bar (-0.121) — where LCZ features actively hurt performance, likely because Singapore's tropical LCZ zones are so different from London's that they introduced noise rather than signal. The four remaining pairs are near zero. The inconsistency itself is the finding — LCZ does not reliably help cross-city generalization.

**Statistical result:** p = 0.670. LCZ and urban features do not produce a statistically significant improvement in cross-city AUC. This is a meaningful negative finding — it challenges the assumption in the literature that urban structural features improve model transferability.

**Interpretation:** The result suggests that LCZ zones encode urban morphology, not climate behavior. When the climate regimes of the train and test city are similar (Singapore→Tokyo), the model already transfers well without LCZ. When they are very different (Singapore→London), LCZ cannot compensate because the problem is atmospheric, not structural. Urban features are useful within cities where rainfall patterns are already known, but they do not bridge climate regime gaps.

---

### Q2: Does Rainfall Memory Transfer Cross-City?

Comparing B_Full_XGB (with rainfall) against B_NoLags_XGB (no rainfall) — same model, same LCZ, only rainfall features differ:

| Pair | With Rainfall AUC | No Rainfall AUC | Benefit |
|------|-----------------|----------------|---------|
| London→New York | 0.974 | 0.651 | +0.323 |
| New York→London | 0.837 | 0.747 | +0.091 |
| Singapore→London | 0.433 | 0.473 | -0.040 |
| Singapore→Tokyo | 0.999 | 0.560 | **+0.440** |
| Tokyo→New York | 0.901 | 0.597 | +0.304 |
| Tokyo→Singapore | 0.989 | 0.612 | +0.376 |

Mean benefit: +0.249 (p = 0.021 — SIGNIFICANT)

**Visualization Interpretation (Q2 Bar Chart)**

The Q2 bar chart is dominated by tall green bars — five out of six pairs show meaningful improvement from including rainfall memory. Singapore→Tokyo has the tallest bar (+0.440), meaning rainfall features are responsible for nearly half the AUC in that pair. The only red bar is Singapore→London (-0.040), but this pair is already near random guessing in both configurations — the model is fundamentally failing regardless of whether rainfall is included. The visual pattern strongly supports the statistical finding.

**Statistical result:** p = 0.021. Rainfall memory significantly improves cross-city performance. This is one of the clearest findings of the study.

**Interpretation:** Rainfall autocorrelation — the fact that today's rainfall predicts tomorrow's — appears to transfer across cities even when urban structure does not. Cities that experience heavy rainfall tend to experience it in sequences, and this pattern is consistent enough across Singapore, Tokyo, London, and New York that a model trained on one city's rainfall sequences can recognize similar patterns in another. This is why removing rainfall features consistently hurts performance while removing LCZ features has no consistent effect.

---

### Q3: Does Building Density Similarity Predict Generalization Quality?

| City | Mean buildings/km² |
|------|--------------------|
| Singapore | 73.41 |
| London | 157.93 |
| New York | 295.86 |
| Tokyo | 785.78 |

Pearson correlation between building density difference and AUC drop: **-0.500**

**Visualization Interpretation (Q3 Scatter Plot)**

The scatter plot shows a counterintuitive negative correlation. Singapore→London (density difference = 84.51, the smallest) clusters at the top-left with the largest AUC drops (0.31–0.57). Singapore→Tokyo and Tokyo→Singapore (density difference = 712.36, the largest) cluster at the bottom-right with near-zero AUC drops. The negative correlation of -0.500 means more different building densities actually associated with smaller drops in this dataset. The scatter makes this visually clear — the cities that are most similar in urban density (Singapore and London) generalize worst to each other, while the most density-different cities (Singapore and Tokyo) transfer almost perfectly.

**Interpretation:** Building density difference is a poor predictor of generalization failure in this study. The actual predictor appears to be climate regime similarity. Singapore and Tokyo share monsoon-influenced intense rainfall despite very different building densities. Singapore and London have more similar building densities but completely different rainfall climatologies. This finding suggests that future cross-city ML studies should prioritize climate classification over urban morphology when selecting training data for generalization.

---

### Statistical Tests Summary

| Test | Comparison | p-value | Result |
|------|-----------|---------|--------|
| Test 1 | A_Full_RF vs B_Full_XGB (LCZ effect) | 0.6697 | NOT SIGNIFICANT |
| Test 2 | B_Full_XGB vs B_NoLags_XGB (rainfall effect) | 0.0211 | SIGNIFICANT |
| Test 3 | Within-city vs cross-city AUC (all models) | 0.0004 | SIGNIFICANT |

**Test 3 is the thesis's central statistical result.** Within-city mean AUC was 0.935. Cross-city mean AUC was 0.767. The difference is statistically proven (p = 0.0004). Standard within-city evaluation significantly overestimates real-world generalization performance. This directly answers the thesis's main research question.

---

## Week 13–14 Key Findings Summary

1. **Within-city evaluation significantly overestimates generalization** — proven statistically (p = 0.0004). Mean AUC drops from 0.935 within-city to 0.767 cross-city.

2. **Generalization is climate-pair dependent, not uniform** — Singapore→Tokyo drops only 0.001 AUC; Singapore→London drops up to 0.567. Climate regime similarity is the primary driver of successful transfer.

3. **LCZ and urban features do not significantly improve cross-city generalization** (p = 0.670). In climatically mismatched pairs, urban features can actively hurt performance. This is a meaningful negative result that challenges assumptions in the urban climate ML literature.

4. **Rainfall memory transfers cross-city and significantly improves generalization** (p = 0.021). Mean AUC improvement of 0.249 from including rainfall features. Rainfall autocorrelation patterns are consistent enough across cities to transfer meaningfully.

5. **Building density similarity does not predict generalization success** (correlation = -0.500, counterintuitive direction). Climate classification is a better guide than urban morphology for selecting training data.

6. **London is consistently the hardest test city** — every model, every training city fails to generalize to London at a high level. Even the best cross-city result for London (New York→London B_Full_XGB, AUC 0.837) represents a substantial drop from within-city performance.

7. **Training direction matters** — London→New York (AUC 0.975) dramatically outperforms New York→London (AUC 0.674) using the same model. Training on a city with more diverse or intense rainfall patterns produces models that generalize better to moderate-rainfall cities than vice versa.

---

*Week 13–14 analysis complete. Week 15 (multi-city training) analysis follows.*

In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from joblib import parallel_backend

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, roc_curve
)
from xgboost import XGBClassifier

# =============================================================
# WEEK 15 — MULTI-CITY TRAINING
# =============================================================
#
# GOAL:
#   Train on 3 cities combined, test on the 4th held-out city.
#   Compare against Week 13 best single-city result for same test city.
#   Question: does multi-city training improve generalization?
#
# SETUP (leave-one-out — each city tested exactly once):
#   Run 1: Train Singapore+London+New_York  → Test Tokyo
#   Run 2: Train Tokyo+London+New_York      → Test Singapore
#   Run 3: Train Tokyo+Singapore+New_York   → Test London
#   Run 4: Train Tokyo+Singapore+London     → Test New_York
#
# MODELS (same as Weeks 13-14 for consistency):
#   1. A_Full_RF     — RF, no urban/LCZ, full rainfall
#   2. B_Full_XGB    — XGB, urban + LCZ + full rainfall
#   3. B_NoLags_XGB  — XGB, urban + LCZ, no rainfall (forecasting)
#
# TRAINING DATA:
#   First 80% of each training city (consistent with all prior experiments)
#
# LCZ ALIGNMENT:
#   Same fill-with-0 strategy as Week 13-14.
#   Training cities' LCZ columns are unioned.
#   Test city missing zones filled with 0.
#   Test city extra zones dropped.
#
# TOKYO SPEED:
#   Any run involving Tokyo uses n_jobs=-1 + parallel backend.
#
# COMPARISON:
#   Week 13 best single-city AUC per test city (from your results)
#   vs Week 15 multi-city AUC — direct improvement table.
#
# OUTPUT:
#   Appended to Results\tables.txt
#   Figures saved to Results\figures\ with week15_ prefix
#   Results\week15_results.csv
#
# =============================================================


# =========================
# 1. PATHS
# =========================

BASE        = r"C:\Users\janaa\UrbanRainfall-ML-Thesis"
RESULTS_DIR = rf"{BASE}\Results"
FIGURES_DIR = rf"{RESULTS_DIR}\figures"
TABLES_FILE = rf"{RESULTS_DIR}\tables.txt"

os.makedirs(FIGURES_DIR, exist_ok=True)

city_files = {
    "Singapore": rf"{BASE}\Data\Processed\Singapore_week7_ready.csv",
    "Tokyo":     rf"{BASE}\Data\Processed\Tokyo_week7_ready.csv",
    "London":    rf"{BASE}\Data\Processed\London_week7_ready.csv",
    "New_York":  rf"{BASE}\Data\Processed\New_York_week7_ready.csv",
}

# Append to existing tables.txt
tables_out = open(TABLES_FILE, "a", encoding="utf-8")

def log(text=""):
    print(text)
    tables_out.write(text + "\n")

def save_fig(name):
    path = rf"{FIGURES_DIR}\week15_{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: week15_{name}.png")


# =========================
# 2. LOAD ALL CITIES
# =========================

city_data = {}
log("\n\n" + "="*70)
log("WEEK 15 — MULTI-CITY TRAINING")
log("="*70)
log("\n=== LOADING CITIES ===\n")

for city, path in city_files.items():
    df = pd.read_csv(path)
    df = df.sort_values(["year", "day_of_year"]).reset_index(drop=True)
    city_data[city] = df
    log(f"{city}: {len(df):,} rows | "
        f"extreme_rain rate: {df['extreme_rain'].mean():.3f} | "
        f"extreme days: {int(df['extreme_rain'].sum()):,}")


# =========================
# 3. FEATURE SETS
# (identical to Weeks 13-14 for consistency)
# =========================

def get_features(df):
    rain_cols    = ["rainfall", "rain_lag_1", "rain_lag_2", "rain_lag_3",
                    "rain_roll_3", "rain_roll_7"]
    weather_cols = ["tavg", "tmin", "tmax", "pres", "wspd"]
    time_cols    = ["month", "day_of_year", "year"]
    geo_cols     = ["x", "y"]

    # Urban + LCZ — B sets only
    urban_cols   = ["city_area_km2", "building_count", "buildings_per_km2",
                    "total_building_area_km2", "building_area_density"]
    lcz_onehot   = [c for c in df.columns if c.startswith("LCZ_")]
                    
    lcz_cols     = ["avg_LCZ_city"] + lcz_onehot
    base_no_urban = weather_cols + time_cols + geo_cols
    urban_lcz     = urban_cols + lcz_cols

    return {
        "A_Full":   rain_cols + base_no_urban,
        "B_Full":   rain_cols + base_no_urban + urban_lcz,
        "B_NoLags": base_no_urban + urban_lcz,
    }


# =========================
# 4. BUILD COMBINED TRAINING SET
# Concatenate 80% of each training city
# Union all LCZ columns — missing ones filled with 0
# =========================

def build_multi_city_train(train_cities, feature_key):
    """
    Combines first 80% of each training city into one training set.
    Takes the union of all LCZ columns across training cities.
    Missing LCZ columns in any city filled with 0.
    Returns combined X_train, y_train with all columns aligned.
    """
    frames = []
    all_lcz_cols = set()

    # First pass: collect all LCZ columns across all training cities
    for city in train_cities:
        df   = city_data[city]
        feat = get_features(df)[feature_key]
        lcz  = [c for c in feat if c.startswith("LCZ_")]
        all_lcz_cols.update(lcz)

    all_lcz_cols = sorted(list(all_lcz_cols))

    # Second pass: build each city's training slice with full LCZ union
    for city in train_cities:
        df        = city_data[city]
        feat      = get_features(df)[feature_key]
        split_idx = int(len(df) * 0.8)

        X = df[feat].iloc[:split_idx].copy()
        y = df["extreme_rain"].iloc[:split_idx]

        # Fill missing LCZ columns with 0
        for col in all_lcz_cols:
            if col not in X.columns:
                X[col] = 0

        frames.append((X, y))

    # Align all frames to same column order
    # Get union of all feature columns
    all_cols = set()
    for X, _ in frames:
        all_cols.update(X.columns)
    all_cols = sorted(list(all_cols))

    aligned_frames = []
    for X, y in frames:
        for col in all_cols:
            if col not in X.columns:
                X[col] = 0
        aligned_frames.append((X[all_cols], y))

    X_train = pd.concat([f[0] for f in aligned_frames], ignore_index=True)
    y_train = pd.concat([f[1] for f in aligned_frames], ignore_index=True)

    print(f"  Combined training set: {len(X_train):,} rows | "
          f"extreme rate: {y_train.mean():.3f} | "
          f"features: {len(X_train.columns)}")

    return X_train, y_train, all_cols


# =========================
# 5. ALIGN TEST CITY TO TRAINING COLUMNS
# Same fill-with-0 strategy as Weeks 13-14
# =========================

def align_test_to_train(X_test, train_cols, test_city, train_cities):
    """
    Aligns test city features to match training column set.
    Missing columns filled with 0.
    Extra columns in test dropped.
    """
    X_test = X_test.copy()

    missing = [c for c in train_cols if c not in X_test.columns]
    extra   = [c for c in X_test.columns if c not in train_cols]

    for col in missing:
        X_test[col] = 0
        if col.startswith("LCZ_"):
            print(f"    {col} missing in {test_city} -> filled with 0")

    if extra:
        X_test = X_test.drop(columns=extra)
        lcz_extra = [c for c in extra if c.startswith("LCZ_")]
        if lcz_extra:
            print(f"    LCZ zones in {test_city} unseen in training: {lcz_extra} -> dropped")

    return X_test[train_cols]


# =========================
# 6. MODEL BUILDERS
# Tokyo involved -> use parallel backend
# =========================

def build_model(model_type, scale_pos_weight, use_tokyo_speed):
    if model_type == "RF":
        return RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1 if use_tokyo_speed else 1
        )
    elif model_type == "XGB":
        return XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            n_jobs=-1 if use_tokyo_speed else 1
        )


# =========================
# 7. MULTI-CITY RUN FUNCTION
# =========================

multi_results = []

def run_multi(train_cities, test_city, feature_key, model_type):
    """
    Trains on combined 80% of all train_cities.
    Tests on FULL test_city.
    Scaler fit on combined training data only.
    """
    use_tokyo_speed = "Tokyo" in train_cities or test_city == "Tokyo"

    print(f"\n  {'+'.join(train_cities)} -> {test_city} | "
          f"{feature_key}_{model_type}")

    # Build combined training set
    X_train, y_train, train_cols = build_multi_city_train(
        train_cities, feature_key
    )

    # Build test set
    df_test  = city_data[test_city]
    feat_test = get_features(df_test)[feature_key]
    X_test   = df_test[feat_test]
    y_test   = df_test["extreme_rain"]

    # Align test to training columns
    X_test = align_test_to_train(X_test, train_cols, test_city, train_cities)

    # Scale — fit on combined training data only
    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Imbalance ratio from combined training data
    spw = (y_train == 0).sum() / (y_train == 1).sum()

    # Build and train
    model = build_model(model_type, spw, use_tokyo_speed)

    if use_tokyo_speed:
        with parallel_backend("loky"):
            model.fit(X_train_s, y_train)
    else:
        model.fit(X_train_s, y_train)

    preds = model.predict(X_test_s)
    probs = model.predict_proba(X_test_s)[:, 1]

    auc  = roc_auc_score(y_test, probs)
    f1   = f1_score(y_test, preds, zero_division=0)
    rec  = recall_score(y_test, preds, zero_division=0)
    prec = precision_score(y_test, preds, zero_division=0)
    acc  = accuracy_score(y_test, preds)

    train_label = "+".join(train_cities)

    multi_results.append({
        "Train":       train_label,
        "Test":        test_city,
        "Model":       f"{feature_key}_{model_type}",
        "Feature":     feature_key,
        "ModelType":   model_type,
        "Accuracy":    round(acc,  6),
        "Precision":   round(prec, 6),
        "Recall":      round(rec,  6),
        "F1":          round(f1,   6),
        "ROC_AUC":     round(auc,  6),
    })

    print(f"  Result: AUC={auc:.4f} | F1={f1:.4f} | Recall={rec:.4f}")
    return model, X_test_s, y_test, preds, probs


# =========================
# 8. RUN ALL 4 LEAVE-ONE-OUT SETUPS x 3 MODELS = 12 EXPERIMENTS
# =========================

all_cities = ["Singapore", "Tokyo", "London", "New_York"]

models_to_run = [
    ("A_Full",   "RF"),
    ("B_Full",   "XGB"),
    ("B_NoLags", "XGB"),
]

log("\n=== RUNNING WEEK 15 — MULTI-CITY TRAINING ===")
log("Leave-one-out: each city tested once, other 3 train together")
log("4 setups x 3 models = 12 experiments\n")

for test_city in all_cities:
    train_cities = [c for c in all_cities if c != test_city]
    log(f"\n--- Test: {test_city} | "
        f"Train: {' + '.join(train_cities)} ---")
    for feature_key, model_type in models_to_run:
        run_multi(train_cities, test_city, feature_key, model_type)


# =========================
# 9. RESULTS TABLE
# =========================

multi_df = pd.DataFrame(multi_results)

log("\n\n" + "="*70)
log("WEEK 15 — MULTI-CITY RESULTS (sorted by ROC_AUC)")
log("="*70)
log(multi_df.sort_values("ROC_AUC", ascending=False).to_string(index=False))

multi_df.to_csv(rf"{RESULTS_DIR}\week15_results.csv", index=False)
log("\nSaved: week15_results.csv")

log("\n--- Average by Model ---")
log(multi_df.groupby("Model")[["Accuracy","Precision","Recall","F1","ROC_AUC"]]
    .mean().round(4).to_string())


# =========================
# 10. COMPARISON TABLE
# Week 13 best single-city vs Week 15 multi-city
# Per test city per model
# =========================

# Week 13 best single-city AUC per test city per model
# (from your Week 13 results — best AUC any single train city achieved)
week13_best = {
    # Test=Tokyo: best was Singapore->Tokyo
    ("Tokyo",     "A_Full_RF"):    0.998748,
    ("Tokyo",     "B_Full_XGB"):   0.999020,
    ("Tokyo",     "B_NoLags_XGB"): 0.559506,

    # Test=Singapore: best was Tokyo->Singapore
    ("Singapore", "A_Full_RF"):    0.983529,
    ("Singapore", "B_Full_XGB"):   0.988588,
    ("Singapore", "B_NoLags_XGB"): 0.612221,

    # Test=London: best was New_York->London
    ("London",    "A_Full_RF"):    0.673683,
    ("London",    "B_Full_XGB"):   0.837476,
    ("London",    "B_NoLags_XGB"): 0.746595,

    # Test=New_York: best was London->New_York
    ("New_York",  "A_Full_RF"):    0.974743,
    ("New_York",  "B_Full_XGB"):   0.973753,
    ("New_York",  "B_NoLags_XGB"): 0.650691,
}

compare_rows = []
for _, row in multi_df.iterrows():
    key    = (row["Test"], row["Model"])
    w13    = week13_best.get(key)
    if w13 is not None:
        diff = round(row["ROC_AUC"] - w13, 4)
        compare_rows.append({
            "Test_City":       row["Test"],
            "Model":           row["Model"],
            "Week13_Best_AUC": w13,
            "Week15_Multi_AUC":row["ROC_AUC"],
            "Improvement":     diff,
            "Direction":       "IMPROVED" if diff > 0.01
                               else "WORSE" if diff < -0.01
                               else "NO CHANGE"
        })

compare_df = pd.DataFrame(compare_rows).sort_values(
    ["Test_City","Model"])

log("\n\n" + "="*70)
log("WEEK 15 — COMPARISON: SINGLE-CITY (W13 BEST) vs MULTI-CITY")
log("="*70)
log("Improvement > 0 = multi-city training helped")
log("Improvement < 0 = single-city training was better\n")
log(compare_df.to_string(index=False))

log("\n--- Summary by Direction ---")
log(compare_df["Direction"].value_counts().to_string())

log("\n--- Average improvement by model ---")
log(compare_df.groupby("Model")["Improvement"].mean().round(4).to_string())

log("\n--- Average improvement by test city ---")
log(compare_df.groupby("Test_City")["Improvement"].mean().round(4).to_string())


# =========================
# 11. STATISTICAL TEST
# Is multi-city AUC significantly different from single-city best?
# Paired t-test across 12 matched pairs
# =========================

log("\n\n" + "="*70)
log("WEEK 15 — STATISTICAL TEST")
log("="*70)
log("Paired t-test: Week 13 best single-city AUC vs Week 15 multi-city AUC")
log("Tests whether multi-city training produces significantly different results\n")

w13_aucs   = list(compare_df["Week13_Best_AUC"])
w15_aucs   = list(compare_df["Week15_Multi_AUC"])

t, p = stats.ttest_rel(w13_aucs, w15_aucs)
log(f"Week 13 mean AUC: {np.mean(w13_aucs):.4f}")
log(f"Week 15 mean AUC: {np.mean(w15_aucs):.4f}")
log(f"t-statistic: {t:.4f}")
log(f"p-value:     {p:.4f}")

if p < 0.05:
    if np.mean(w15_aucs) > np.mean(w13_aucs):
        log("Result: SIGNIFICANT improvement (p < 0.05)")
        log("Multi-city training significantly improves generalization.")
    else:
        log("Result: SIGNIFICANT degradation (p < 0.05)")
        log("Multi-city training significantly hurts performance.")
else:
    log("Result: NOT SIGNIFICANT (p >= 0.05)")
    log("Multi-city training does not significantly change performance")
    log("compared to the best single-city training.")


# =============================================================
# VISUALIZATIONS
# =============================================================

print("\n=== GENERATING WEEK 15 FIGURES ===\n")

# FIG 1: Multi-city AUC heatmap
# Rows = test city, Cols = model, Color = AUC
pivot_multi = multi_df.pivot_table(
    index="Test", columns="Model", values="ROC_AUC"
)
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_multi, annot=True, fmt=".3f",
            cmap="RdYlGn", vmin=0.5, vmax=1.0,
            linewidths=0.5, annot_kws={"size": 12}, ax=ax)
ax.set_title("Week 15 — Multi-City Training ROC-AUC\n"
             "Rows = held-out test city | Cols = model", fontsize=12)
ax.set_xlabel("Model")
ax.set_ylabel("Test City")
plt.tight_layout()
save_fig("multicity_auc_heatmap")

# FIG 2: Grouped bar — Week 13 best vs Week 15 per test city per model
test_cities = all_cities
models_list = ["A_Full_RF", "B_Full_XGB", "B_NoLags_XGB"]
x     = np.arange(len(test_cities))
width = 0.13

fig, ax = plt.subplots(figsize=(14, 5))
colors_w13 = ["#3498db", "#2980b9", "#1a5276"]
colors_w15 = ["#e67e22", "#d35400", "#7d3c98"]

for i, model_name in enumerate(models_list):
    w13_vals = [week13_best.get((city, model_name), 0) for city in test_cities]
    w15_vals = [
        multi_df[(multi_df["Test"] == city) &
                 (multi_df["Model"] == model_name)]["ROC_AUC"].values[0]
        if len(multi_df[(multi_df["Test"] == city) &
                        (multi_df["Model"] == model_name)]) > 0
        else 0
        for city in test_cities
    ]
    ax.bar(x + (i*2)*width,   w13_vals, width,
           label=f"W13 {model_name}", color=colors_w13[i], alpha=0.85)
    ax.bar(x + (i*2+1)*width, w15_vals, width,
           label=f"W15 {model_name}", color=colors_w15[i], alpha=0.85)

ax.set_xticks(x + 2.5*width)
ax.set_xticklabels(test_cities, fontsize=10)
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.3, 1.05)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8,
           label="Random baseline")
ax.set_title("Week 13 Best Single-City vs Week 15 Multi-City\n"
             "Blue = Week 13 | Orange/Purple = Week 15")
ax.legend(fontsize=7, ncol=3)
plt.tight_layout()
save_fig("w13_vs_w15_comparison")

# FIG 3: Improvement bar chart (positive = multi helped, negative = hurt)
fig, ax = plt.subplots(figsize=(11, 4))
colors = ["#2ecc71" if v > 0.01 else "#e74c3c" if v < -0.01 else "#95a5a6"
          for v in compare_df["Improvement"]]
x = range(len(compare_df))
ax.bar(x, compare_df["Improvement"], color=colors)
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r['Test_City']}\n{r['Model']}" for _, r in compare_df.iterrows()],
    rotation=45, ha="right", fontsize=7)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("AUC Change (Multi - Single)")
ax.set_title("Week 15 — Did Multi-City Training Help?\n"
             "Green = improved | Red = worse | Grey = no change (±0.01)")
plt.tight_layout()
save_fig("improvement_over_single_city")

# FIG 4: ROC curves — all 3 models for London test city
# (London was the hardest in Week 13 — most interesting to see improvement)
log_models_for_roc = ["A_Full_RF", "B_Full_XGB", "B_NoLags_XGB"]
test_city_roc      = "London"
train_cities_roc   = [c for c in all_cities if c != test_city_roc]

fig, ax = plt.subplots(figsize=(7, 6))

for feature_key, model_type in models_to_run:
    model_name    = f"{feature_key}_{model_type}"
    use_tokyo     = True  # London test always has Tokyo in training
    X_train, y_train, train_cols = build_multi_city_train(
        train_cities_roc, feature_key)

    df_test   = city_data[test_city_roc]
    feat_test = get_features(df_test)[feature_key]
    X_test    = df_test[feat_test]
    y_test    = df_test["extreme_rain"]
    X_test    = align_test_to_train(
        X_test, train_cols, test_city_roc, train_cities_roc)

    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    spw   = (y_train == 0).sum() / (y_train == 1).sum()
    model = build_model(model_type, spw, use_tokyo)
    with parallel_backend("loky"):
        model.fit(X_train_s, y_train)

    probs    = model.predict_proba(X_test_s)[:, 1]
    auc_val  = multi_df[(multi_df["Test"] == test_city_roc) &
                        (multi_df["Model"] == model_name)]["ROC_AUC"].values
    auc_val  = auc_val[0] if len(auc_val) > 0 else roc_auc_score(y_test, probs)
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.3f})")

ax.plot([0,1],[0,1], "--", color="gray", label="Random baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Week 15 — ROC Curves (Test City: {test_city_roc})\n"
             f"Trained on: {' + '.join(train_cities_roc)}")
ax.legend(fontsize=9)
plt.tight_layout()
save_fig(f"roc_curves_{test_city_roc}_test")


# =========================
# FINAL SUMMARY
# =========================

log("\n\n" + "="*70)
log("WEEK 15 FINAL SUMMARY")
log("="*70)

log(f"\nTotal experiments: {len(multi_df)} (4 test cities x 3 models)")

log("\n--- Best multi-city result per model ---")
log(multi_df.loc[multi_df.groupby("Model")["ROC_AUC"].idxmax()][
    ["Model","Test","ROC_AUC","F1","Recall"]].to_string(index=False))

log("\n--- Worst multi-city result per model ---")
log(multi_df.loc[multi_df.groupby("Model")["ROC_AUC"].idxmin()][
    ["Model","Test","ROC_AUC","F1","Recall"]].to_string(index=False))

log("\n--- Did multi-city training help? (vs Week 13 best) ---")
log(compare_df[["Test_City","Model","Week13_Best_AUC",
                "Week15_Multi_AUC","Improvement","Direction"]]
    .to_string(index=False))

log(f"\nOverall: {(compare_df['Direction']=='IMPROVED').sum()} improved | "
    f"{(compare_df['Direction']=='WORSE').sum()} worse | "
    f"{(compare_df['Direction']=='NO CHANGE').sum()} no change")

log(f"\nStatistical test p-value: {p:.4f} — "
    f"{'SIGNIFICANT' if p < 0.05 else 'NOT SIGNIFICANT'}")

tables_out.close()
print(f"\nAll tables appended to: {TABLES_FILE}")
print(f"All figures saved to:   {FIGURES_DIR}")
print("\n=== WEEK 15 DONE ===")



WEEK 15 — MULTI-CITY TRAINING

=== LOADING CITIES ===

Singapore: 657,120 rows | extreme_rain rate: 0.050 | extreme days: 32,873
Tokyo: 4,408,180 rows | extreme_rain rate: 0.050 | extreme days: 220,530
London: 651,644 rows | extreme_rain rate: 0.050 | extreme days: 32,600
New_York: 525,696 rows | extreme_rain rate: 0.050 | extreme days: 26,300

=== RUNNING WEEK 15 — MULTI-CITY TRAINING ===
Leave-one-out: each city tested once, other 3 train together
4 setups x 3 models = 12 experiments


--- Test: Singapore | Train: Tokyo + London + New_York ---

  Tokyo+London+New_York -> Singapore | A_Full_RF
  Combined training set: 4,468,415 rows | extreme rate: 0.051 | features: 16
  Result: AUC=0.9972 | F1=0.9626 | Recall=0.9279

  Tokyo+London+New_York -> Singapore | B_Full_XGB
  Combined training set: 4,468,415 rows | extreme rate: 0.051 | features: 38
    LCZ_0.0 missing in Singapore -> filled with 0
    LCZ_1.0 missing in Singapore -> filled with 0
    LCZ_15.0 missing in Singapore -> fille

# Week 16 — Multi-City Training Analysis
## Month 4 Results: Week 15

**Thesis:** Evaluating the Generalization of Machine Learning Models for Urban Extreme Rainfall Prediction
**Setup:** Leave-one-out — each city tested once, other 3 cities train together
**Models:** A_Full_RF · B_Full_XGB · B_NoLags_XGB
**Experiments:** 4 test cities × 3 models = 12 experiments

---

## Overview

Week 15 addresses a natural follow-up question raised by the Week 13–14 findings: if single-city training fails to generalize to some cities, does training on multiple cities simultaneously improve performance? A leave-one-out approach was used — each city was held out as the test city exactly once while the remaining three cities provided the combined training data. This is the standard and most symmetric approach for this type of evaluation, ensuring no city is treated differently from another.

The combined training datasets were substantial. When Tokyo was included in training, the combined set reached over 4.4 million rows. The 80% split was applied to each contributing city before combining, maintaining methodological consistency with all prior experiments.

---

## Week 15 Results

### Multi-City AUC Results

| Test City | Train Cities | A_Full_RF | B_Full_XGB | B_NoLags_XGB |
|-----------|-------------|-----------|------------|--------------|
| Singapore | Tokyo + London + New York | 0.997 | 0.999 | 0.593 |
| Tokyo | Singapore + London + New York | 0.999 | 0.998 | 0.656 |
| London | Singapore + Tokyo + New York | 0.671 | 0.832 | 0.783 |
| New York | Singapore + Tokyo + London | 0.991 | 0.991 | 0.728 |

Average AUC by model:
- A_Full_RF: 0.914
- B_Full_XGB: 0.955
- B_NoLags_XGB: 0.690

### Week 13 Best vs Week 15 Multi-City Comparison

| Test City | Model | Week 13 Best AUC | Week 15 AUC | Change | Direction |
|-----------|-------|-----------------|------------|--------|-----------|
| London | A_Full_RF | 0.674 | 0.671 | -0.003 | NO CHANGE |
| London | B_Full_XGB | 0.837 | 0.832 | -0.006 | NO CHANGE |
| London | B_NoLags_XGB | 0.747 | 0.783 | **+0.037** | IMPROVED |
| New York | A_Full_RF | 0.975 | 0.991 | **+0.016** | IMPROVED |
| New York | B_Full_XGB | 0.974 | 0.991 | **+0.017** | IMPROVED |
| New York | B_NoLags_XGB | 0.651 | 0.728 | **+0.077** | IMPROVED |
| Singapore | A_Full_RF | 0.984 | 0.997 | **+0.014** | IMPROVED |
| Singapore | B_Full_XGB | 0.989 | 0.999 | **+0.011** | IMPROVED |
| Singapore | B_NoLags_XGB | 0.612 | 0.593 | -0.020 | WORSE |
| Tokyo | A_Full_RF | 0.999 | 0.999 | +0.000 | NO CHANGE |
| Tokyo | B_Full_XGB | 0.999 | 0.998 | -0.001 | NO CHANGE |
| Tokyo | B_NoLags_XGB | 0.560 | 0.656 | **+0.097** | IMPROVED |

Overall: 7 improved · 4 no change · 1 worse

Statistical test (paired t-test, Week 13 vs Week 15):
- Week 13 mean AUC: 0.833
- Week 15 mean AUC: 0.853
- p-value: 0.0712 — NOT SIGNIFICANT

---

## Visualization Interpretation

### Multi-City AUC Heatmap

The heatmap shows test city on the rows and model on the columns. Three rows — New York, Singapore, Tokyo — are dark green across A_Full_RF and B_Full_XGB, confirming strong multi-city generalization for these cities. London remains the clear outlier — its row is orange-yellow for A_Full_RF (0.671), light green for B_Full_XGB (0.832), and yellow-green for B_NoLags_XGB (0.783). Even with training data from three diverse cities including Tokyo's massive 4.4 million row dataset, London cannot be predicted reliably by the simpler RF model. The B_NoLags_XGB column is notably the weakest across all test cities, with Singapore (0.593) shown in orange-red — the only truly poor result in Week 15. This heatmap makes immediately clear that London is the persistent problem city and that no-lag forecasting without rainfall features remains fundamentally limited regardless of how much training data is provided.

### ROC Curves — London Test City

London was chosen for the ROC curve visualization because it was the hardest test city throughout the study. The three curves tell a revealing story. A_Full_RF (blue, AUC 0.671) performs the worst — its curve rises slowly and stays close to the diagonal for much of its range, indicating poor discrimination ability even with full rainfall features and three cities of training data. B_Full_XGB (orange, AUC 0.832) rises more steeply in the early section, showing better ability to catch true extreme events at low false positive rates — this is the most operationally useful profile for a warning system. B_NoLags_XGB (green, AUC 0.783) sits between the two — surprisingly close to B_Full_XGB despite having no rainfall features at all, suggesting that for London specifically, weather and urban structure features combined with diverse multi-city training provide meaningful signal even without rainfall memory. The fact that B_NoLags_XGB nearly matches B_Full_XGB for London is a notable reversal from Week 13 where B_NoLags consistently underperformed — multi-city training appears to partially compensate for the absence of rainfall features specifically for this difficult city.

### Week 13 vs Week 15 Comparison Bar Chart

The grouped bar chart directly compares Week 13 best single-city AUC (blue shades) against Week 15 multi-city AUC (orange/purple shades) for each test city. For Singapore and Tokyo, the bars are nearly identical in height — multi-city training made almost no difference because single-city training was already near perfect for these cities. For New York, all three Week 15 bars are visibly taller than their Week 13 counterparts, particularly for B_NoLags_XGB where the purple bar clearly exceeds the dark blue bar. For London, the picture is mixed — A_Full_RF bars are essentially the same height, B_Full_XGB bars are also nearly identical, but B_NoLags_XGB shows a visible gain (0.747 to 0.783). The chart makes clear that multi-city training primarily benefits cities that were already struggling and that the gains are concentrated in the no-lag forecasting scenario.

### Improvement Bar Chart (Multi - Single)

The improvement chart shows green bars (multi-city helped), one red bar (multi-city hurt), and grey bars (no change). The tallest green bar is Tokyo B_NoLags_XGB (+0.097) — the largest single improvement in the study — followed by New York B_NoLags_XGB (+0.077) and London B_NoLags_XGB (+0.037). The pattern is visually obvious: almost all the green bars belong to B_NoLags_XGB. The only red bar is Singapore B_NoLags_XGB (-0.020), a marginal degradation. The grey bars — no meaningful change — are concentrated in the full-feature models for Tokyo and London's A_Full_RF. This chart communicates the core Week 15 finding more clearly than any table: multi-city training helps the forecasting scenario (no-lag) more than the full-feature scenario, and helps cities that were already struggling more than cities that were already performing well.

---

## Key Findings — Week 15

**1. Multi-city training produces modest, consistent improvement — but not a breakthrough**

7 out of 12 comparisons improved, 4 showed no change, and only 1 got marginally worse. The statistical test (p = 0.0712) falls just short of significance. The overall picture is positive but not transformative. Mean AUC improved from 0.833 to 0.853 — a real gain, but not large enough to fundamentally change the generalization landscape established in Week 13.

**2. B_NoLags_XGB benefits most from multi-city training**

Average improvement for B_NoLags_XGB was 0.048 AUC, compared to 0.007 for A_Full_RF and 0.005 for B_Full_XGB. This makes intuitive sense — when a model has no rainfall features to rely on, it must learn weather and urban patterns. Seeing more diverse cities during training gives it more weather pattern examples to generalize from. Multi-city training partially compensates for the absence of rainfall memory.

**3. London remains the hardest city — even with 3 cities training together**

A_Full_RF on London: 0.671 in Week 15 vs 0.674 in Week 13 — essentially unchanged. Even adding Singapore, Tokyo, and New York as training data could not improve the RF model's ability to predict London's extreme rain. B_Full_XGB did slightly better (0.832) but still significantly below the performance seen on other test cities. This strongly reinforces the climate regime argument — London's temperate maritime rainfall patterns are sufficiently different from the other three cities that no amount of additional training data from those cities can close the gap.

**4. Tokyo and Singapore — already near ceiling, no room to improve**

For Tokyo and Singapore with full features, Week 13 single-city results were already near 0.999. Multi-city training cannot improve on near-perfect performance. Any marginal changes in these rows are noise rather than signal. The ceiling effect means multi-city training is most valuable for cities where single-city transfer was genuinely failing.

**5. New York shows the clearest across-the-board improvement**

New York improved across all three models — +0.016, +0.017, and +0.077. This suggests New York's rainfall patterns have enough in common with the diverse training set (Singapore, Tokyo, London) that the combined exposure genuinely helps. New York's continental climate with clear seasonal structure may be easier to approximate from diverse training data than London's consistently moderate maritime patterns.

**6. The practical recommendation from Week 15**

For cities where single-city cross-city transfer already works well (Singapore↔Tokyo), multi-city training adds no value. For cities where single-city transfer struggles (London, and forecasting scenarios generally), multi-city training provides modest improvement — particularly for no-lag forecasting. The recommendation is therefore context-dependent: multi-city training is most worthwhile when the target city is climatically dissimilar from any single available training city, and when operating in a forecasting scenario without access to real-time rainfall data.

---

## Week 15 Summary

| Metric | Value |
|--------|-------|
| Total experiments | 12 |
| Improved vs Week 13 | 7 / 12 |
| No change | 4 / 12 |
| Worse | 1 / 12 |
| Mean AUC improvement | +0.020 |
| Statistical significance | p = 0.071 (not significant) |
| Biggest beneficiary model | B_NoLags_XGB (+0.048 avg) |
| Biggest beneficiary city | New York (+0.037 avg) |
| Persistent problem city | London (RF unchanged at 0.671) |

---

*Week 15 analysis complete. Week 16 full cross-phase comparison (Weeks 9–15) follows.*

# Week 16 — Full Cross-Phase Comparison & Month 4 Summary
## Weeks 9–15: Within-City → Cross-City → Multi-City

**Thesis:** Evaluating the Generalization of Machine Learning Models for Urban Extreme Rainfall Prediction
**Cities:** Singapore · London · New York · Tokyo
**Models:** Logistic Regression (LR) · Random Forest (RF) · XGBoost (XGB)
**Total experiments:** 72 within-city (Weeks 9–12) + 18 cross-city (Week 13) + 12 multi-city (Week 15) = 102 experiments

---

## Overview

This document synthesizes all experimental results across the three evaluation phases of the thesis. The progression from within-city evaluation to single cross-city transfer to multi-city training represents the core research arc: establishing a performance baseline, demonstrating its limitations, and testing a potential remedy. The comparison across all three phases directly answers the thesis's main research question and all three sub-questions.

---

## Phase 1 — Within-City Evaluation (Weeks 9–12)

### Best Model Results Per City

The following table shows the best-performing model configuration for each city in the within-city evaluation, using the models carried forward into cross-city testing:

| City | A_Full_RF AUC | B_Full_XGB AUC | B_NoLags_XGB AUC |
|------|--------------|----------------|------------------|
| Singapore | 1.000 | 0.9999 | 0.783 |
| London | 1.000 | 0.9999 | 0.819 |
| New York | 1.000 | 0.9999 | 0.836 |
| Tokyo | 1.000 | 0.9999 | 0.808 |
| **Mean** | **1.000** | **0.9999** | **0.811** |

### Full Within-City Results Summary (All 18 Configurations, All 4 Cities)

**Singapore:**

| Model | AUC | F1 | Recall |
|-------|-----|-----|--------|
| A_Full_RF | 1.000 | 0.9999 | 0.9999 |
| B_Full_XGB | 0.9999 | 0.982 | 0.991 |
| A_NoLags_XGB | 0.800 | 0.353 | 0.333 |
| B_NoLags_XGB | 0.783 | 0.320 | 0.295 |
| A_Lags_RF | 0.827 | 0.130 | 0.072 |
| B_NoLags_RF | 0.785 | 0.228 | 0.135 |

**London:**

| Model | AUC | F1 | Recall |
|-------|-----|-----|--------|
| A_Full_RF | 1.000 | 0.9999 | 0.9998 |
| B_Full_XGB | 0.9999 | 0.992 | 0.993 |
| B_Lags_XGB | 0.839 | 0.241 | 0.263 |
| A_Lags_RF | 0.824 | 0.000 | 0.000 |
| B_NoLags_XGB | 0.819 | 0.247 | 0.345 |
| A_NoLags_RF | 0.810 | 0.002 | 0.001 |

**New York:**

| Model | AUC | F1 | Recall |
|-------|-----|-----|--------|
| A_Full_RF | 1.000 | 1.000 | 1.000 |
| B_Full_XGB | 0.9999 | 0.994 | 0.996 |
| B_Lags_LR | 0.858 | 0.245 | 0.807 |
| B_Lags_RF | 0.854 | 0.028 | 0.014 |
| B_NoLags_XGB | 0.836 | 0.249 | 0.346 |
| A_NoLags_RF | 0.830 | 0.044 | 0.023 |

**Tokyo:**

| Model | AUC | F1 | Recall |
|-------|-----|-----|--------|
| A_Full_RF | 1.000 | 1.000 | 1.000 |
| B_Full_XGB | 0.9999 | 0.985 | 1.000 |
| B_NoLags_XGB | 0.808 | 0.263 | 0.469 |
| A_NoLags_XGB | 0.806 | 0.270 | 0.472 |
| B_Lags_RF | 0.788 | 0.000 | 0.000 |
| A_Lags_RF | 0.781 | 0.000 | 0.000 |

### Within-City Key Findings

Within-city evaluation produced near-perfect or perfect results across all four cities for full-feature models. A_Full_RF achieved AUC = 1.000 in every city. B_Full_XGB achieved AUC ≥ 0.9999 in every city. Even Logistic Regression reached AUC = 1.000 in all cities with full features. This consistency across four geographically and climatically distinct cities confirms that within-city evaluation is not a meaningful test of model capability when same-day rainfall is included.

Feature importance analysis revealed that rainfall alone accounted for approximately 70–80% of model importance across all cities. This was the most consistent finding in the entire study — identical feature hierarchy in Singapore, London, New York, and Tokyo. The models are not learning complex urban climate relationships. They are learning rainfall autocorrelation: high rainfall today predicts the extreme rainfall classification.

The no-lag scenario (B_NoLags_XGB) produced AUC of 0.783–0.836 within cities — substantially lower than full-feature models but still meaningful. This is the forecasting scenario where no same-day rainfall is available, and it represents the realistic operational context for an early warning system.

---

## Phase 2 — Single Cross-City Evaluation (Week 13–14)

### Cross-City Results by Test City

| Test City | Best Train City | A_Full_RF AUC | B_Full_XGB AUC | B_NoLags_XGB AUC |
|-----------|----------------|--------------|----------------|------------------|
| Singapore | Tokyo | 0.984 | 0.989 | 0.612 |
| Tokyo | Singapore | 0.999 | 0.999 | 0.560 |
| London | New York | 0.674 | 0.837 | 0.747 |
| New York | London | 0.975 | 0.974 | 0.651 |
| **Mean** | — | **0.908** | **0.950** | **0.643** |

### Performance Drop: Within-City vs Best Cross-City

| City | Model | Within AUC | Best Cross AUC | Drop |
|------|-------|-----------|---------------|------|
| Singapore | A_Full_RF | 1.000 | 0.984 | 0.016 |
| Singapore | B_Full_XGB | 0.9999 | 0.989 | 0.011 |
| Singapore | B_NoLags_XGB | 0.783 | 0.612 | 0.171 |
| Tokyo | A_Full_RF | 1.000 | 0.999 | 0.001 |
| Tokyo | B_Full_XGB | 0.9999 | 0.999 | 0.001 |
| Tokyo | B_NoLags_XGB | 0.808 | 0.560 | 0.248 |
| London | A_Full_RF | 1.000 | 0.674 | **0.326** |
| London | B_Full_XGB | 0.9999 | 0.837 | **0.163** |
| London | B_NoLags_XGB | 0.819 | 0.747 | 0.072 |
| New York | A_Full_RF | 1.000 | 0.975 | 0.025 |
| New York | B_Full_XGB | 0.9999 | 0.974 | 0.026 |
| New York | B_NoLags_XGB | 0.836 | 0.651 | 0.185 |

Statistical confirmation (Test 3, paired t-test):
- Within-city mean AUC: 0.935
- Cross-city mean AUC: 0.767
- p = 0.0004 — **SIGNIFICANT**

### Cross-City Key Findings

The performance drop was statistically proven (p = 0.0004), directly confirming the thesis central argument that within-city evaluation significantly overestimates real-world generalization performance.

The drop was highly pair-dependent — Singapore→Tokyo lost only 0.001 AUC while Singapore→London lost 0.446–0.567. Climate regime similarity emerged as the primary driver of successful generalization, not geographic proximity or urban structural similarity.

LCZ and urban features did not significantly improve cross-city performance (p = 0.670). In some pairs, adding LCZ features actively degraded performance. Rainfall memory, however, significantly improved cross-city AUC (p = 0.021), with a mean benefit of 0.249.

---

## Phase 3 — Multi-City Training (Week 15)

### Multi-City Results

| Test City | A_Full_RF AUC | B_Full_XGB AUC | B_NoLags_XGB AUC |
|-----------|--------------|----------------|------------------|
| Singapore | 0.997 | 0.999 | 0.593 |
| Tokyo | 0.999 | 0.998 | 0.656 |
| London | 0.671 | 0.832 | 0.783 |
| New York | 0.991 | 0.991 | 0.728 |
| **Mean** | **0.914** | **0.955** | **0.690** |

Statistical test (Week 13 best vs Week 15):
- p = 0.0712 — NOT SIGNIFICANT

---

## Full Three-Phase Comparison

### AUC Across All Three Phases — By Test City and Model

#### A_Full_RF

| Test City | Within-City | Cross-City (W13) | Multi-City (W15) | W13 Drop | W15 vs W13 |
|-----------|------------|-----------------|-----------------|---------|-----------|
| Singapore | 1.000 | 0.984 | 0.997 | -0.016 | +0.013 |
| Tokyo | 1.000 | 0.999 | 0.999 | -0.001 | +0.000 |
| London | 1.000 | 0.674 | 0.671 | -0.326 | -0.003 |
| New York | 1.000 | 0.975 | 0.991 | -0.025 | +0.016 |
| **Mean** | **1.000** | **0.908** | **0.914** | **-0.092** | **+0.007** |

#### B_Full_XGB

| Test City | Within-City | Cross-City (W13) | Multi-City (W15) | W13 Drop | W15 vs W13 |
|-----------|------------|-----------------|-----------------|---------|-----------|
| Singapore | 0.9999 | 0.989 | 0.999 | -0.011 | +0.011 |
| Tokyo | 0.9999 | 0.999 | 0.998 | -0.001 | -0.001 |
| London | 0.9999 | 0.837 | 0.832 | -0.163 | -0.006 |
| New York | 0.9999 | 0.974 | 0.991 | -0.026 | +0.017 |
| **Mean** | **0.9999** | **0.950** | **0.955** | **-0.050** | **+0.005** |

#### B_NoLags_XGB

| Test City | Within-City | Cross-City (W13) | Multi-City (W15) | W13 Drop | W15 vs W13 |
|-----------|------------|-----------------|-----------------|---------|-----------|
| Singapore | 0.783 | 0.612 | 0.593 | -0.171 | -0.020 |
| Tokyo | 0.808 | 0.560 | 0.656 | -0.248 | +0.097 |
| London | 0.819 | 0.747 | 0.783 | -0.072 | +0.037 |
| New York | 0.836 | 0.651 | 0.728 | -0.185 | +0.077 |
| **Mean** | **0.811** | **0.643** | **0.690** | **-0.169** | **+0.048** |

---

## Cross-Phase Narrative Analysis

### The Three-Phase Story

**Phase 1 sets the ceiling.** Within-city evaluation produces near-perfect performance for every full-feature model in every city. This is the "optimistic picture" the thesis argues against. An RF or XGB model evaluated only within its training city would appear deployment-ready. AUC = 1.000 across four cities with three model types sends an unambiguous signal that the task is solved. But it is not solved — it is merely well-memorized.

**Phase 2 reveals the reality.** Moving models to unseen cities collapses performance in ways that are dramatic, inconsistent, and city-pair dependent. The overall mean AUC for A_Full_RF drops from 1.000 to 0.908 — but that average conceals a range from 0.554 to 0.999. London as a test city drops A_Full_RF to 0.674. This is the central finding: within-city AUC is not a reliable predictor of cross-city AUC, and the gap is statistically proven (p = 0.0004).

**Phase 3 tests the fix.** Multi-city training partially addresses the generalization gap but does not eliminate it. Mean AUC improves modestly from 0.908 to 0.914 for A_Full_RF and from 0.643 to 0.690 for B_NoLags_XGB. London remains the hardest city at 0.671 even with three cities of training data. The fix is real but incomplete, and not statistically significant at p = 0.0712.

---

### What Changed Across Phases by Model

**A_Full_RF** — starts perfect within-city, drops sharply cross-city for London and moderately for New York, partially recovers with multi-city training for all cities except London. The pattern reveals RF's dependence on rainfall features that behave consistently only within the same climate regime. When crossing climate regimes, RF has no fallback — it over-relies on rainfall autocorrelation that does not generalize.

**B_Full_XGB** — follows a similar trajectory but degrades less severely cross-city (mean drop 0.050 vs 0.092 for RF). XGBoost with LCZ and urban features is the most consistent cross-city model. The average cross-city AUC of 0.950 and multi-city AUC of 0.955 make it the strongest overall performer when generalization is the goal. Importantly, B_Full_XGB is also the only model where adding LCZ features occasionally helps cross-city — New York→London improved by +0.164 with B_Full_XGB vs A_Full_RF.

**B_NoLags_XGB** — the forecasting scenario model. Starts lower within-city (mean 0.811), drops the most cross-city (mean 0.643), but benefits the most from multi-city training (mean +0.048 improvement). This pattern makes intuitive sense: without rainfall features, the model must rely on weather and urban patterns that are more generalizable when the training set includes diverse city examples. Multi-city training partially substitutes for the missing rainfall signal by providing broader weather pattern exposure.

---

### London: The Persistent Problem City

London deserves specific attention across all three phases as it is the city where every approach fails most consistently.

| Phase | A_Full_RF | B_Full_XGB | B_NoLags_XGB |
|-------|-----------|------------|--------------|
| Within-city | 1.000 | 0.9999 | 0.819 |
| Best cross-city (W13) | 0.674 | 0.837 | 0.747 |
| Multi-city (W15) | 0.671 | 0.832 | 0.783 |

Three observations stand out. First, A_Full_RF is essentially unchanged from Week 13 to Week 15 for London (0.674 vs 0.671) — adding training data from three more cities including Tokyo's massive dataset makes no difference. Second, B_Full_XGB reaches 0.837 in both Week 13 and Week 15 — LCZ and urban features help London specifically, but multi-city training adds nothing on top. Third, B_NoLags_XGB shows the only meaningful London improvement from multi-city training (0.747 to 0.783) — and this is the model with no rainfall features, suggesting that diverse weather patterns from multi-city training provide London-relevant signal that individual cities could not.

The London pattern confirms the climate regime hypothesis. London's temperate maritime rainfall is structurally different from Singapore's tropical monsoon, Tokyo's typhoon-influenced patterns, and New York's continental precipitation. No machine learning approach in this study — single-city training, LCZ features, multi-city training — could fully bridge that gap. The gap is atmospheric, not structural, and the features available in this study do not capture the atmospheric dynamics that differentiate London's extreme rainfall mechanism from the other three cities.

---

### The Role of LCZ Across All Three Phases

LCZ features were a central hypothesis of the thesis — the expectation being that standardized urban typology would improve cross-city transfer by giving models information about urban structure that transcends city-specific rainfall patterns.

The results across all three phases do not support this hypothesis in its strong form:

**Within-city (Weeks 9–12):** LCZ features added negligible improvement. B sets performed comparably to A sets. Feature importance confirmed LCZ columns ranked near the bottom in all cities.

**Cross-city (Week 13–14):** LCZ did not significantly improve generalization (p = 0.670). In Singapore→London, adding LCZ actively hurt performance (-0.121 AUC). In New York→London, it helped (+0.164). The effect was inconsistent and pair-dependent.

**Multi-city (Week 15):** B_Full_XGB slightly outperformed A_Full_RF on average (0.955 vs 0.914) but the gap reflects both model type (XGB vs RF) and LCZ features — it is not possible to isolate LCZ alone in this comparison.

The conclusion is that LCZ features encode urban morphology but not atmospheric dynamics. Since extreme rainfall prediction is fundamentally an atmospheric problem, urban structure features provide limited leverage for cross-city transfer. This is a meaningful contribution to the literature — studies that have found LCZ useful (Sharma et al., 2025; Bechtel et al., 2023) focused on urban heat, where building morphology directly influences temperature. Rainfall extremes have a different causal chain where LCZ is less relevant.

---

### Rainfall Memory Across All Three Phases

The B_NoLags_XGB comparison across phases provides the clearest lens on the role of rainfall memory:

| Phase | Mean AUC (B_NoLags_XGB) |
|-------|------------------------|
| Within-city | 0.811 |
| Cross-city (W13) | 0.643 |
| Multi-city (W15) | 0.690 |

Rainfall memory matters at every phase. Within-city, removing rainfall features drops AUC by approximately 0.19 compared to full-feature models. Cross-city, the no-lag model drops an additional 0.168 from its within-city baseline. Multi-city training partially recovers this loss (+0.048) but does not close the gap.

Statistical confirmation: rainfall memory significantly improves cross-city performance (p = 0.021, mean benefit 0.249 AUC). This is the most actionable finding of the study — for an operational forecasting system that cannot wait for same-day rainfall measurements, XGBoost with multi-city training and LCZ features (B_NoLags_XGB, Week 15) is the recommended configuration, achieving a mean AUC of 0.690 across held-out test cities.

---

## Research Questions — Answered

**Main Research Question: How accurately can supervised ML models generalize across multiple urban contexts to predict extreme rainfall?**

Accuracy varies substantially by city pair and evaluation phase. Within-city AUC averages 1.000 for full-feature RF, creating an illusion of solved performance. Cross-city AUC for the same model averages 0.908 — a statistically significant drop (p = 0.0004) — with individual pairs ranging from 0.554 to 0.999. Multi-city training recovers some of this loss, reaching a mean of 0.914 for RF and 0.955 for XGB with LCZ. In the forecasting scenario without rainfall features, cross-city AUC averages 0.643 improving to 0.690 with multi-city training.

**Sub-Question 1: How do urban characteristics influence cross-city generalization performance?**

Urban features (LCZ, building density) did not significantly improve cross-city generalization (p = 0.670). Building density difference showed a counterintuitive negative correlation with AUC drop (-0.500) — cities with more different building densities did not generalize worse. Climate regime similarity was a stronger predictor of generalization success than urban morphology similarity. LCZ features helped in specific pairs (New York→London) but hurt in others (Singapore→London), producing no consistent benefit.

**Sub-Question 2: Do standard evaluation metrics adequately capture differences in performance between cities?**

No. The London within-city A_Lags_RF achieved 95.1% accuracy with F1 = 0.000 — it never detected a single extreme event. Singapore→London B_Full_XGB achieved AUC = 0.433 cross-city despite perfect within-city performance. These results demonstrate that accuracy inflated by class imbalance and AUC measured only within-city both provide misleading pictures of real-world model reliability. F1, recall, and cross-city AUC together provide a more complete and honest evaluation framework.

---

## Month 4 Summary — Key Findings

1. **Within-city evaluation significantly overestimates generalization** (p = 0.0004). Mean AUC drops from 0.935 within-city to 0.767 cross-city. This is the thesis's central statistical proof.

2. **Generalization failure is climate-pair dependent.** Singapore↔Tokyo transfer almost perfectly (AUC drop < 0.002). Singapore→London collapses to AUC 0.433–0.554. The range of outcomes across pairs is wider than the range across models.

3. **LCZ and urban features do not reliably improve cross-city generalization** (p = 0.670). The hypothesis that standardized urban typology aids transfer is not supported for rainfall prediction. Urban structure features are more relevant for heat-related predictions where the physical mechanism directly involves building morphology.

4. **Rainfall memory significantly transfers cross-city** (p = 0.021). Rainfall autocorrelation patterns are consistent enough across different cities and climate regimes to provide meaningful cross-city signal. Rainfall features should be retained in any operational cross-city deployment where same-day measurements are available.

5. **Multi-city training provides modest, consistent improvement** — 7 of 12 comparisons improved, mean AUC +0.020, but p = 0.071 falls just short of significance. The benefit is concentrated in the no-lag forecasting scenario and in cities that were already struggling in Week 13.

6. **London is persistently the hardest city** — resistant to all three approaches. Even training on 4.4 million rows from three other cities cannot bring A_Full_RF above 0.671. This confirms that climate regime mismatch creates a ceiling that more data alone cannot raise.

7. **B_Full_XGB is the best cross-city model** — mean AUC 0.950 in Week 13 and 0.955 in Week 15, with the most consistent performance across all city pairs. For operational deployment with rainfall features available, XGBoost with LCZ and urban features is the recommended model.

8. **B_NoLags_XGB with multi-city training is the best forecasting model** — mean AUC 0.690 in Week 15, with the largest relative improvement from multi-city training (+0.048). For deployment without real-time rainfall access, this configuration represents the best available option within this study's scope.

---

## Recommendations for Future Research

The findings of this study point to several directions that could improve cross-city generalization beyond what was achieved here:

**Climate regime grouping for training data selection.** Rather than training on all available cities regardless of climate type, future studies could group cities by Köppen climate classification before selecting training data. Training Singapore together with other humid tropical cities and London together with other temperate oceanic cities would likely produce better within-regime generalization than the mixed-regime approach used here.

**Absolute rather than city-specific thresholds for extreme rainfall.** Defining extreme rainfall relative to each city's own distribution (top 5%) creates threshold mismatches across cities. A model trained on Singapore's extreme threshold may not recognize London's extreme events, which involve much lower absolute rainfall amounts. Future work could explore absolute thresholds (e.g., >30mm/day regardless of city) to improve label consistency across training sets.

**Additional atmospheric features.** The feature set used in this study (temperature, pressure, wind speed) represents surface-level meteorology. Including atmospheric moisture indices, convective available potential energy (CAPE), or humidity profiles would give models access to the physics that actually drive extreme precipitation — features that are physically meaningful across all cities regardless of urban structure.

**Domain adaptation techniques.** Specialized transfer learning or domain adaptation methods could explicitly train models to learn city-invariant features. Unlike the standard approach used here, these methods are designed to handle distribution shifts between training and test environments — precisely the challenge posed by cross-city deployment.

---

*Week 16 analysis complete. All experimental phases documented.*
*Month 5 — thesis writing — begins with Weeks 17–20.*